In [1]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
import pinecone
from sentence_transformers import SentenceTransformer

In [2]:
files = pd.read_csv("course_descriptions.csv", encoding = "ANSI")

In [3]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
            the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [4]:
pd.set_option('display.max_rows', 106)
files['course_description_new'] = files.apply(create_course_description, axis = 1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is sql,\n    ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is sta...
14     The course name is Mathematics, the slug is ma...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is pr...
17     The course name is Start

In [5]:
%load_ext dotenv
%dotenv

In [9]:
load_dotenv(find_dotenv(), override = True)

True

In [6]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [7]:
index_name = "my-index"
dimension = 768
metric = "cosine"

In [8]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index succesfully deleted.


In [9]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-rc40est.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",


In [10]:
index = pc.Index(index_name)

## Embedding the data

In [11]:
model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [12]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

In [13]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

In [14]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vectors_to_upsert)

print("Data upserted to Pinecone index")

Data upserted to Pinecone index


## Semantic search

In [19]:
query = "regression"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [20]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12, 
    include_values = True
)

In [21]:
query_results

QueryResponse(matches=[{'id': 'Machine Learning with Ridge and Lasso Regression',
 'score': 0.396452963,
 'values': [-0.0315290578,
            0.0173235144,
            0.0516472571,
            0.0463465042,
            0.0290109534,
            0.00202576816,
            -0.0045281779,
            -0.0031817886,
            -0.0237998907,
            -0.0115958415,
            0.000533805345,
            0.0448181853,
            0.0168482885,
            -0.0363454297,
            -0.0114185596,
            0.0463515408,
            0.042754095,
            -0.0243540127,
            0.0401916727,
            0.0573106892,
            0.0320074596,
            0.0309487376,
            -0.015013882,
            0.0170590989,
            -0.0111454679,
            0.0221162681,
            -0.00823555049,
            0.0646713302,
            -0.0347860307,
            0.0295450166,
            -0.039023675,
            -0.0185041279,
            -0.0184999406,
            -0.003800

In [22]:
score_threshold = 0.3
for match in query_results["matches"]:
    if match['score'] >= score_threshold:
        print(f"Matched item ID: {match['id']}, score: {match['score']}")

Matched item ID: Machine Learning with Ridge and Lasso Regression, score: 0.396452963
